# my-agents Association LoRA Cloud Trainer (Colab + Google Drive)

1. Colab ランタイムを **GPU（推奨: L4、大容量実験時は A100）** に変更します。
2. 事前に Mac 側で `export_dataset_bundle.py` を実行し、`datasets/<DATASET_NAME>/` を Google Drive にアップロードしておきます。
3. 下のパラメータセルを設定して上から順に実行すると、Drive 上の `runs/<RUN_NAME>/` に `peft_adapter/`・`merged_bf16/`・`run_manifest.json` が出力されます。

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/aiagate/my-agents.git"
REPO_BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive/my-agents-training")
DATASET_NAME = "v1_239"
CONFIG_NAME = "qwen25_3b_l4.yaml"
RUN_NAME = "qwen25-3b-v1"

REPO_DIR = Path("/content/my-agents")
DATASET_DIR = DRIVE_ROOT / "datasets" / DATASET_NAME
OUTPUT_DIR = DRIVE_ROOT / "runs" / RUN_NAME
print(f"Dataset: {DATASET_DIR}")
print(f"Output:  {OUTPUT_DIR}")

In [ ]:
import subprocess
import sys

subprocess.run(["nvidia-smi"], check=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

req_path = REPO_DIR / "training" / "association_lora" / "requirements-cuda.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_path)],
    check=True,
)

In [ ]:
import os

try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("HF_TOKEN not set; proceeding without Hugging Face auth.")

In [ ]:
import subprocess
import sys

train_script = (
    REPO_DIR / "training" / "association_lora" / "scripts" / "train_cuda_lora.py"
)
config_path = REPO_DIR / "training" / "association_lora" / "configs" / CONFIG_NAME

subprocess.run(
    [
        sys.executable,
        str(train_script),
        "--config",
        str(config_path),
        "--dataset-dir",
        str(DATASET_DIR),
        "--output-dir",
        str(OUTPUT_DIR),
    ],
    check=True,
)

In [ ]:
import json

manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, ensure_ascii=False, indent=2))

local_mlx_dir = f"~/.local/share/my-agents/mlx-lora/{RUN_NAME}-4bit"
print("\n=== Next commands on Mac ===")
print(
    "uv run --extra mlx python training/association_lora/scripts/prepare_local_mlx.py "
    f"convert --hf-path ~/Downloads/{RUN_NAME}/merged_bf16 "
    f"--mlx-path {local_mlx_dir}"
)
print(
    "uv run --extra mlx python training/association_lora/scripts/eval_local_mlx.py "
    f"--model {local_mlx_dir}"
)